In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("/content/hospital_readmission_risk_10000.csv")
df = df.dropna()
df = df.drop_duplicates()
target_col = "readmission_risk"
X = df.drop(target_col, axis=1)
y = df[target_col]
if y.dtype == 'object':
    le_y = LabelEncoder()
    y = le_y.fit_transform(y)

for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])


X = X.values


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

input_dim = X_train.shape[1]
num_classes = len(np.unique(y))

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model1 = create_model()

history1 = model1.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

loss1, acc1 = model1.evaluate(X_test, y_test)
print("\nModel 1 Test Accuracy:", acc1)


plt.plot(history1.history['loss'], label='Train Loss')
plt.plot(history1.history['val_loss'], label='Val Loss')
plt.title("Model 1 Learning Curve")
plt.legend()
plt.show()


def create_dropout_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model2 = create_dropout_model()

history2 = model2.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

loss2, acc2 = model2.evaluate(X_test, y_test)
print("\nModel 2 (Dropout) Test Accuracy:", acc2)


plt.plot(history2.history['loss'], label='Train Loss')
plt.plot(history2.history['val_loss'], label='Val Loss')
plt.title("Model 2 (Dropout) Learning Curve")
plt.legend()
plt.show()


kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores1 = []
scores2 = []

for train_idx, test_idx in kf.split(X):
    tf.keras.backend.clear_session()

    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]


    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    X_te = sc.transform(X_te)

    # Model 1
    m1 = create_model()
    m1.fit(X_tr, y_tr, epochs=20, batch_size=32, verbose=0)
    _, a1 = m1.evaluate(X_te, y_te, verbose=0)
    scores1.append(a1)


    m2 = create_dropout_model()
    m2.fit(X_tr, y_tr, epochs=20, batch_size=32, verbose=0)
    _, a2 = m2.evaluate(X_te, y_te, verbose=0)
    scores2.append(a2)

print("\nANN Model 1 CV Accuracy:", np.mean(scores1))
print("ANN Model 2 (Dropout) CV Accuracy:", np.mean(scores2))
lr = LogisticRegression(max_iter=300)

cv_scores = cross_val_score(lr, X, y, cv=5)

print("\nLogistic Regression CV Scores:", cv_scores)
print("Logistic Regression Avg Accuracy:", np.mean(cv_scores))

print("\n========== FINAL RESULTS ==========")
print("Model 1 Test Accuracy:", acc1)
print("Model 2 Test Accuracy:", acc2)
print("Model 1 CV Accuracy:", np.mean(scores1))
print("Model 2 CV Accuracy:", np.mean(scores2))
print("Logistic Regression Avg:", np.mean(cv_scores))
models = ['Basic ANN', 'Dropout ANN', 'Logistic Regression']
accuracies = [acc1, acc2, np.mean(cv_scores)]

plt.bar(models, accuracies)
plt.title("Model Performance Comparison")
plt.ylabel("Accuracy")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/hospital_readmission_risk_10000.csv'